In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 

#model
from sklearn.model_selection import train_test_split,GridSearchCV,RandomizedSearchCV 
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score,classification_report

In [2]:
df=pd.read_csv('heart.csv')
display(df.head())
print("***************************************************")
print(df.shape)
print("***************************************************")
print(df.info)
print("***************************************************")
display(df.describe())


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


***************************************************
(918, 12)
***************************************************
<bound method DataFrame.info of      Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  \
0     40   M           ATA        140          289          0     Normal   
1     49   F           NAP        160          180          0     Normal   
2     37   M           ATA        130          283          0         ST   
3     48   F           ASY        138          214          0     Normal   
4     54   M           NAP        150          195          0     Normal   
..   ...  ..           ...        ...          ...        ...        ...   
913   45   M            TA        110          264          0     Normal   
914   68   M           ASY        144          193          1     Normal   
915   57   M           ASY        130          131          0     Normal   
916   57   F           ATA        130          236          0        LVH   
917   38   M      

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


In [3]:
y=df['HeartDisease']
x=df.drop('HeartDisease',axis=1)
x.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up


In [4]:
cat_c=[]
num_c=[]
for col in x.columns : 
    if(x[col].nunique()<15) : 
        cat_c.append(col)
    else :
        num_c.append(col) 
df_cat=x[cat_c]
df_num=x[num_c]
df_cat.head()

,Sex,ChestPainType,FastingBS,RestingECG,ExerciseAngina,ST_Slope
0,M,ATA,0,Normal,N,Up
1,F,NAP,0,Normal,N,Flat
2,M,ATA,0,ST,N,Up
3,F,ASY,0,Normal,Y,Flat
4,M,NAP,0,Normal,N,Up


In [5]:
x_cat=pd.get_dummies(df_cat,drop_first=True)
x_cat=x_cat.astype(int)
x_cat.head()

,FastingBS,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,0,1,1,0,0,1,0,0,0,1
1,0,0,0,1,0,1,0,0,1,0
2,0,1,1,0,0,0,1,0,0,1
3,0,0,0,0,0,1,0,1,1,0
4,0,1,0,1,0,1,0,0,0,1


In [6]:
x=pd.concat([x_cat,df_num],axis=1)
x.head()

,FastingBS,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up,Age,RestingBP,Cholesterol,MaxHR,Oldpeak
0,0,1,1,0,0,1,0,0,0,1,40,140,289,172,0.0
1,0,0,0,1,0,1,0,0,1,0,49,160,180,156,1.0
2,0,1,1,0,0,0,1,0,0,1,37,130,283,98,0.0
3,0,0,0,0,0,1,0,1,1,0,48,138,214,108,1.5
4,0,1,0,1,0,1,0,0,0,1,54,150,195,122,0.0


In [7]:
#train_test 
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2)
print(f"train data shape for x = {xtrain.shape}  test data shape for x = {xtest.shape} ")
print(f"train data shape for y = {ytrain.shape}  test data shape for y = {ytest.shape} ")

train data shape for x = (734, 15)  test data shape for x = (184, 15) 
train data shape for y = (734,)  test data shape for y = (184,) 


In [8]:
#manually spliting datas
row_sampling=xtrain.sample(frac=0.75,replace=True)
col_sampling=xtrain.sample(frac=0.8,replace=True,axis=1)
row_col_sample=xtrain.sample(frac=0.8,replace=True,axis=1).sample(frac=0.75,replace=True)
print(f" row sample = {row_sampling.shape} col sample = {col_sampling.shape}  both sample = {row_col_sample.shape}")

 row sample = (550, 15) col sample = (734, 12)  both sample = (550, 12)


In [9]:
#MODELING 
reg=RandomForestClassifier(n_estimators=80,max_features=4,max_depth=10)
reg.fit(xtrain,ytrain)
print(f"Accuracy score = {accuracy_score(ytest,reg.predict(xtest))}")
print(classification_report(ytest,reg.predict(xtest)))

Accuracy score = 0.8804347826086957
              precision    recall  f1-score   support

           0       0.90      0.86      0.88        94
           1       0.86      0.90      0.88        90

    accuracy                           0.88       184
   macro avg       0.88      0.88      0.88       184
weighted avg       0.88      0.88      0.88       184



In [10]:
param = {
    'n_estimators' : [80,100,200],
    'max_features': ['sqrt', 'log2', None],
    'max_depth': [3, 9, None],
    'min_samples_split': [2, 5, 10]

}

In [11]:
#RandomCV 
rand_cv=RandomizedSearchCV(
    estimator=RandomForestClassifier(),
    param_distributions=param,
    n_iter=10,
    cv=5,
    scoring='accuracy',
    n_jobs=-1

)
rand_cv.fit(xtrain, ytrain)

print(f"Best Random Search Accuracy: {rand_cv.best_score_ * 100:.2f}%")
print(f"Best Parameters: {rand_cv.best_params_}") 

Best Random Search Accuracy: 86.51%
Best Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': 9}


In [15]:
param_grid = {
    'n_estimators': [50,100,200, 300],
    'max_depth': [10,20],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': ['sqrt','log2']
}
grid_search=GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(xtrain, ytrain)
print(f"Best Grid Search Accuracy: {grid_search.best_score_ * 100:.2f}%")
print(f"Best Parameters: {grid_search.best_params_}")

Best Grid Search Accuracy: 86.79%
Best Parameters: {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 50}


In [21]:
dt_ready=pd.concat([x,y],axis=1)
dt_ready.to_excel("heart_ready.xlsx",index=False)